<div style="
background:linear-gradient(135deg,#171922,#10131b);
border:2px solid #3ce7e7;
border-radius:22px;
padding:32px;
font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;
color:white;
box-shadow:0 0 18px rgba(60,231,231,.18);
">

<div style="display:flex;align-items:center;gap:14px;">
<span style="font-size:34px;">👨‍💻</span>

<div>
<div style="
font-size:34px;
font-weight:800;
color:#4ff5f5;
text-shadow:0 0 8px rgba(79,245,245,.35);
">
Aliakbar Isapour
</div>

<div style="
color:#bfbfbf;
font-size:16px;
margin-top:3px;
">
Machine Learning • Data Science • AI
</div>
</div>
</div>

<div style="
height:2px;
background:linear-gradient(90deg,#7b3ff2,#4ff5f5);
border-radius:5px;
margin:22px 0;
"></div>

<div style="
color:#d7d7d7;
font-size:16px;
line-height:1.7;
margin-bottom:25px;
">

</div>


<div style="display:flex;gap:14px;flex-wrap:wrap;">

<a href="https://github.com/isapour0"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
GitHub
</a>

<a href="https://www.linkedin.com/in/aliakbar-isapour-682592413"
target="_blank"
rel="noopener noreferrer"
style="
display:inline-block;
margin: 0 10px;
text-decoration:none;
background:#23283a;
color:#4ff5f5;
padding:12px 22px;
border-radius:12px;
border:1px solid #4ff5f5;
font-weight:bold;
transition:0.25s;
">
LinkedIn
</a>


</div>

</div>

<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">📌 Library Imports</h2>
<hr style="border:1px solid #8A2BE2; opacity:.4;">
</div>


In [3]:
# ---- core ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
import warnings

# ---- modelSelection and metrics ----
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ---- keras ----
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

classes_name = [
    'Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
    'Dog', 'Frog', 'Horse', 'Ship', 'Truck'
]

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-radius:18px;
border-left: 8px solid #00E5FF;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">
    
<h2 style="color:#00E5FF;">📂 Data Loding</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">
</div>


In [4]:
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

paths = {name: DATA_DIR / f"{name}.npy" for name in ["X_train", "y_train", "X_test", "y_test"]}

if all(p.exists() for p in paths.values()):
    print("Loading CIFAR-10 from local .npy files...")
    X_train = np.load(paths["X_train"])
    y_train = np.load(paths["y_train"])
    X_test = np.load(paths["X_test"])
    y_test = np.load(paths["y_test"])
else:
    print("Downloading CIFAR-10 with Keras...")
    (X_train, y_train), (X_test, y_test) = cifar10.load_data()
    np.save(paths["X_train"], X_train)
    np.save(paths["y_train"], y_train)
    np.save(paths["X_test"], X_test)
    np.save(paths["y_test"], y_test)
    print("Saved to", DATA_DIR.resolve())

Loading CIFAR-10 from local .npy files...


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">🔧 Preprocessing</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">

<ul>
    
<li>Step 1: Separating a 10% validation set.</li> 
<li>Step 2: Scaling pixels to the range [0, 1].</li>
<li>Step 3: Step 3: Encoding tags using One hot Encoding.</li> 

</ul>
</div>

In [5]:
# Step 1: Separating a 10% validation set
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    stratify=y_train.ravel(),
    random_state=RANDOM_STATE,
)

print("Train part:", X_train_part.shape, "| Validation:", X_val.shape, "| Test:", X_test.shape)

Train part: (45000, 32, 32, 3) | Validation: (5000, 32, 32, 3) | Test: (10000, 32, 32, 3)


In [6]:
# Step 2: Scaling pixels to the range [0, 1]
def prepare_images(X):
    """uint8 images (N, 32, 32, 3) -> float32 in [0, 1], flattened to (N, 3072)."""
    return (X.astype("float32") / 255.0)

X_train_flat = prepare_images(X_train_part)
X_val_flat = prepare_images(X_val)
X_test_flat = prepare_images(X_test)

for name, arr in [("X_train_flat", X_train_flat), ("X_val_flat", X_val_flat), ("X_test_flat", X_test_flat)]:
    print(f"{name:12s} shape={arr.shape}  dtype={arr.dtype}  min={arr.min():.1f}  max={arr.max():.1f}")

X_train_flat shape=(45000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0
X_val_flat   shape=(5000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0
X_test_flat  shape=(10000, 32, 32, 3)  dtype=float32  min=0.0  max=1.0


In [7]:
# Step 3: Encoding tags using One hot Encoding
y_train_cat = to_categorical(y_train_part, num_classes=10)
y_val_cat = to_categorical(y_val, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print("y_train_cat:", y_train_cat.shape, "| y_val_cat:", y_val_cat.shape, "| y_test_cat:", y_test_cat.shape)

y_train_cat: (45000, 10) | y_val_cat: (5000, 10) | y_test_cat: (10000, 10)


In [8]:
pd.DataFrame({
    "train_part": np.bincount(y_train_part.ravel(), minlength=10),
    "validation": np.bincount(y_val.ravel(), minlength=10),
}, index=classes_name)

,train_part,validation
Airplane,4500,500
Automobile,4500,500
Bird,4500,500
Cat,4500,500
Deer,4500,500
Dog,4500,500
Frog,4500,500
Horse,4500,500
Ship,4500,500
Truck,4500,500


<div style="
background:linear-gradient(135deg,#08131f,#0d1b2a,#132238);
border:1px solid #00d4ff;
border-left: 8px solid #00E5FF;
border-radius:18px;
padding:25px;
box-shadow:0 0 25px rgba(0,212,255,.25);
color:white;
font-family:Segoe UI;
">

<h2 style="color:#00E5FF;">🤖 CNN Model</h2>
<hr style="border:1px solid #8A2BE2;opacity:.4;">

<!-- <li>Step 1: Build an all-connected network</li>
<li>Step 2: Compile with learning rete, evaluation criteria and CrossEntropy</li>
<li>Step 3: Fitting on training data validation date sets</li> -->
</div>